In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('data.csv')

In [3]:
df.sample(5)

,Location,Area,Baths,Beds,Dining Room,Laundry Room,Store Rooms,Kitchens,Drawing Room,Gym,Powder Room,Steam Room,No additional rooms,Prayer Rooms,Lounge or Sitting Room,Price
3477,"North Nazimabad, Karachi, Sindh",256 Sq. Yd.,6,6,1,1,1,2,1,1,1,1,0,1,1,PKR5.25 Crore
6032,"DHA Defence, Islamabad, Islamabad Capital",10 Marla,5,5,1,0,1,2,1,0,0,0,0,0,0,PKR5.95 Crore
4915,"Gadap Town, Karachi, Sindh",120 Sq. Yd.,5,4,1,1,1,2,1,0,0,0,0,0,1,PKR2.05 Crore
7216,"G-13, Islamabad, Islamabad Capital",8 Marla,6,6,1,1,1,1,1,0,1,1,0,1,1,PKR5.5 Crore
4783,"DHA Defence, Karachi, Sindh",100 Sq. Yd.,4,3,0,1,1,1,1,0,1,0,0,0,1,PKR3.55 Crore


In [4]:
df.shape

(7500, 16)

In [5]:
df["Price"].nunique()

629

In [6]:
df.isnull().sum()

Location                  0
Area                      0
Baths                     0
Beds                      0
Dining Room               0
Laundry Room              0
Store Rooms               0
Kitchens                  0
Drawing Room              0
Gym                       0
Powder Room               0
Steam Room                0
No additional rooms       0
Prayer Rooms              0
Lounge or Sitting Room    0
Price                     0
dtype: int64

In [7]:
# Let's first examine the unique values in the Price column to understand the format
print("Unique price values:")
print(df['Price'].value_counts().head(10))
print(f"\nTotal unique price values: {df['Price'].nunique()}")


Unique price values:
Price
PKR7.5 Crore     122
PKR3.5 Crore     113
PKR2.5 Crore      95
PKR9.5 Crore      94
PKR8.5 Crore      93
PKR1.55 Crore     90
PKR10.5 Crore     88
PKR4.5 Crore      84
PKR2.6 Crore      84
PKR8 Crore        78
Name: count, dtype: int64

Total unique price values: 629


In [8]:
# Function to convert price to numeric
def convert_price_to_numeric(price_str):
    """
    Convert price string like 'PKR6.5 Crore' to numeric value in PKR
    """
    if pd.isna(price_str):
        return np.nan
    
    # Remove 'PKR' prefix
    price_str = str(price_str).replace('PKR', '').strip()
    
    # Extract numeric part and unit
    import re
    match = re.match(r'([\d.]+)\s*([A-Za-z]+)', price_str)
    
    if match:
        numeric_value = float(match.group(1))
        unit = match.group(2).lower()
        
        # Convert to base PKR based on unit
        if 'crore' in unit:
            return numeric_value * 10000000  # 1 crore = 10 million
        elif 'lakh' in unit:
            return numeric_value * 100000    # 1 lakh = 100 thousand
        elif 'thousand' in unit or 'k' in unit:
            return numeric_value * 1000
        else:
            return numeric_value  # Assume it's already in base PKR
    else:
        return np.nan

# Apply the conversion
df['Price_Numeric'] = df['Price'].apply(convert_price_to_numeric)

# Display the results
print("Sample of converted prices:")
print(df[['Price', 'Price_Numeric']].head(10))


Sample of converted prices:
           Price  Price_Numeric
0   PKR3.5 Crore     35000000.0
1  PKR1.32 Crore     13200000.0
2  PKR1.74 Crore     17400000.0
3  PKR10.3 Crore    103000000.0
4  PKR8.25 Crore     82500000.0
5    PKR15 Crore    150000000.0
6     PKR85 Lakh      8500000.0
7     PKR63 Lakh      6300000.0
8   PKR8.9 Crore     89000000.0
9    PKR17 Crore    170000000.0


In [9]:
# Check for any conversion issues
print("Conversion summary:")
print(f"Total rows: {len(df)}")
print(f"Successfully converted: {df['Price_Numeric'].notna().sum()}")
print(f"Failed conversions: {df['Price_Numeric'].isna().sum()}")

# Show any failed conversions
if df['Price_Numeric'].isna().sum() > 0:
    print("\nFailed conversions:")
    failed_prices = df[df['Price_Numeric'].isna()]['Price'].unique()
    for price in failed_prices:
        print(f"  - {price}")

# Basic statistics of the numeric price column
print(f"\nPrice statistics (in PKR):")
print(f"Min: {df['Price_Numeric'].min():,.0f}")
print(f"Max: {df['Price_Numeric'].max():,.0f}")
print(f"Mean: {df['Price_Numeric'].mean():,.0f}")
print(f"Median: {df['Price_Numeric'].median():,.0f}")


Conversion summary:
Total rows: 7500
Successfully converted: 7500
Failed conversions: 0

Price statistics (in PKR):
Min: 1
Max: 970,000,000
Mean: 80,712,326
Median: 49,500,000


In [10]:
# Optional: If you want to replace the original Price column with the numeric version
# Uncomment the following lines:

# df['Price'] = df['Price_Numeric']
# df = df.drop('Price_Numeric', axis=1)

# Or if you want to keep both columns, you can rename them for clarity:
# df = df.rename(columns={'Price': 'Price_Original', 'Price_Numeric': 'Price'})

print("DataFrame info after conversion:")
print(df.info())


DataFrame info after conversion:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7500 entries, 0 to 7499
Data columns (total 17 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Location                7500 non-null   object 
 1   Area                    7500 non-null   object 
 2   Baths                   7500 non-null   object 
 3   Beds                    7500 non-null   object 
 4   Dining Room             7500 non-null   int64  
 5   Laundry Room            7500 non-null   int64  
 6   Store Rooms             7500 non-null   int64  
 7   Kitchens                7500 non-null   int64  
 8   Drawing Room            7500 non-null   int64  
 9   Gym                     7500 non-null   int64  
 10  Powder Room             7500 non-null   int64  
 11  Steam Room              7500 non-null   int64  
 12  No additional rooms     7500 non-null   int64  
 13  Prayer Rooms            7500 non-null   int64  
 14  Lounge 

In [11]:
df.sample(5)

,Location,Area,Baths,Beds,Dining Room,Laundry Room,Store Rooms,Kitchens,Drawing Room,Gym,Powder Room,Steam Room,No additional rooms,Prayer Rooms,Lounge or Sitting Room,Price,Price_Numeric
4686,"Bahria Town Karachi, Karachi, Sindh",200 Sq. Yd.,3,3,1,0,1,1,1,0,0,0,0,0,0,PKR1.65 Crore,16500000.0
5735,"G-13, Islamabad, Islamabad Capital",7 Marla,6,6,1,1,0,2,1,0,1,1,0,0,1,PKR5.8 Crore,58000000.0
6464,"I-8, Islamabad, Islamabad Capital",1.2 Kanal,6,6,1,1,1,2,1,1,1,1,0,1,1,PKR20 Crore,200000000.0
4095,"Cantt, Karachi, Sindh",375 Sq. Yd.,5,5,0,0,1,1,0,0,0,0,0,0,0,PKR7.65 Crore,76500000.0
4004,"Bahria Town Karachi, Karachi, Sindh",79 Sq. Yd.,3,3,0,0,0,0,0,0,0,0,1,0,0,PKR71.23 Lakh,7123000.0


In [12]:
df["Price_Numeric"].nunique()

629

In [13]:
# Target Encodding Location


In [14]:
encoded = df.groupby('Location')['Price_Numeric'].mean()
print(encoded)


Location
AGHOSH, Islamabad, Islamabad Capital                            5.375000e+07
Aaqa Society, Lahore, Punjab                                    2.000000e+07
Abdalians Cooperative Housing Society, Lahore, Punjab           1.500000e+07
Airline Housing Society, Lahore, Punjab                         6.300000e+07
Airport Avenue Housing Society, Islamabad, Islamabad Capital    3.000000e+07
                                                                    ...     
Wapda Town, Lahore, Punjab                                      4.476154e+07
West Wood Housing Society, Lahore, Punjab                       7.250000e+07
Youhanabad, Lahore, Punjab                                      1.950000e+07
Zamzama, Karachi, Sindh                                         2.206250e+08
Zaraj Housing Scheme, Islamabad, Islamabad Capital              3.925000e+07
Name: Price_Numeric, Length: 267, dtype: float64


In [15]:
df['Location'].nunique()

267

In [16]:
encoded.nunique()

227

In [17]:
df["Location_Encoded"] = df["Location"].map(encoded)

In [18]:
df.sample(5)

,Location,Area,Baths,Beds,Dining Room,Laundry Room,Store Rooms,Kitchens,Drawing Room,Gym,Powder Room,Steam Room,No additional rooms,Prayer Rooms,Lounge or Sitting Room,Price,Price_Numeric,Location_Encoded
109,"DHA Defence, Lahore, Punjab",1 Kanal,6,5,1,0,1,2,1,0,1,0,0,0,1,PKR8.5 Crore,85000000.0,8.656734e+07
5391,"G-13, Islamabad, Islamabad Capital",10 Marla,6,7,1,1,0,2,1,1,1,1,0,1,1,PKR8.25 Crore,82500000.0,6.460466e+07
932,"DHA Defence, Lahore, Punjab",2 Kanal,6,5,1,1,2,2,1,0,1,1,0,0,0,PKR16 Crore,160000000.0,8.656734e+07
6013,"G-13, Islamabad, Islamabad Capital",10 Marla,6,6,0,0,0,0,0,0,0,0,0,0,0,PKR7.4 Crore,74000000.0,6.460466e+07
3449,"DHA Defence, Karachi, Sindh",244 Sq. Yd.,3,3,0,0,1,2,0,0,0,0,0,0,0,PKR5.75 Crore,57500000.0,1.498673e+08


In [19]:
df["Area"].unique()

array(['10 Marla', '4 Marla', '5 Marla', '1 Kanal', '2 Kanal', '17 Marla',
       '8 Marla', '3.1 Kanal', '14.5 Marla', '3 Marla', '1.1 Kanal',
       '3.5 Marla', '5.2 Marla', '5.8 Marla', '11 Marla', '3 Kanal',
       '1.5 Marla', '2.8 Marla', '2.9 Kanal', '1.2 Kanal', '2 Marla',
       '1.2 Marla', '4.5 Marla', '9 Marla', '6 Marla', '13 Marla',
       '12 Marla', '4 Kanal', '12.5 Marla', '10.5 Marla', '10.6 Marla',
       '7 Marla', '20 Kanal', '1.7 Kanal', '1.9 Kanal', '2.5 Marla',
       '2.1 Kanal', '5 Kanal', '12.7 Marla', '2.7 Marla', '10.8 Marla',
       '1.5 Kanal', '1.8 Marla', '11.5 Marla', '10.7 Marla', '14 Marla',
       '1.3 Kanal', '6.5 Marla', '8.5 Marla', '8.2 Marla', '18 Marla',
       '2.5 Kanal', '5.5 Marla', '16 Marla', '6.4 Marla', '3.2 Marla',
       '15 Marla', '9.5 Marla', '1.8 Kanal', '2.6 Kanal', '5.3 Marla',
       '11.9 Marla', '10.9 Marla', '1.6 Kanal', '3.5 Kanal', '3.8 Marla',
       '7.1 Marla', '4.8 Marla', '0.2 Marla', '2.1 Marla', '2.2 Marla',
     

In [20]:
# Function to convert area to square feet
def convert_area_to_sqft(area_str):
    """
    Convert area string to square feet
    Conversion factors:
    - 1 Marla = 272.25 sq ft (standard conversion)
    - 1 Kanal = 20 Marla = 5445 sq ft
    - 1 Sq. Yd. = 9 sq ft
    """
    if pd.isna(area_str):
        return np.nan
    
    area_str = str(area_str).strip()
    
    # Remove commas for large numbers
    area_str = area_str.replace(',', '')
    
    # Extract numeric part and unit
    import re
    match = re.match(r'([\d.]+)\s*([A-Za-z\s.]+)', area_str)
    
    if match:
        numeric_value = float(match.group(1))
        unit = match.group(2).strip().lower()
        
        # Convert to square feet based on unit
        if 'marla' in unit:
            return numeric_value * 272.25  # 1 Marla = 272.25 sq ft
        elif 'kanal' in unit:
            return numeric_value * 5445    # 1 Kanal = 5445 sq ft (20 Marla)
        elif 'sq. yd.' in unit or 'sq yd' in unit or 'square yard' in unit:
            return numeric_value * 9       # 1 Sq. Yd. = 9 sq ft
        elif 'sq ft' in unit or 'sqft' in unit or 'square feet' in unit:
            return numeric_value           # Already in sq ft
        else:
            return np.nan  # Unknown unit
    else:
        return np.nan

# Apply the conversion
df['Area_SqFt'] = df['Area'].apply(convert_area_to_sqft)

# Display the results
print("Sample of converted areas:")
print(df[['Area', 'Area_SqFt']].head(10))


Sample of converted areas:
       Area  Area_SqFt
0  10 Marla    2722.50
1   4 Marla    1089.00
2   5 Marla    1361.25
3   1 Kanal    5445.00
4   1 Kanal    5445.00
5   2 Kanal   10890.00
6   4 Marla    1089.00
7   5 Marla    1361.25
8  17 Marla    4628.25
9   2 Kanal   10890.00


In [21]:
# Check for any conversion issues
print("Area conversion summary:")
print(f"Total rows: {len(df)}")
print(f"Successfully converted: {df['Area_SqFt'].notna().sum()}")
print(f"Failed conversions: {df['Area_SqFt'].isna().sum()}")

# Show any failed conversions
if df['Area_SqFt'].isna().sum() > 0:
    print("\nFailed conversions:")
    failed_areas = df[df['Area_SqFt'].isna()]['Area'].unique()
    for area in failed_areas:
        print(f"  - {area}")

# Basic statistics of the numeric area column
print(f"\nArea statistics (in square feet):")
print(f"Min: {df['Area_SqFt'].min():,.0f} sq ft")
print(f"Max: {df['Area_SqFt'].max():,.0f} sq ft")
print(f"Mean: {df['Area_SqFt'].mean():,.0f} sq ft")
print(f"Median: {df['Area_SqFt'].median():,.0f} sq ft")


Area conversion summary:
Total rows: 7500
Successfully converted: 7500
Failed conversions: 0

Area statistics (in square feet):
Min: 54 sq ft
Max: 108,900 sq ft
Mean: 3,740 sq ft
Median: 2,722 sq ft


In [22]:
# Show examples of different area types and their conversions
print("Examples of area conversions:")
examples = df[['Area', 'Area_SqFt']].drop_duplicates().head(15)
for idx, row in examples.iterrows():
    print(f"{row['Area']:15} -> {row['Area_SqFt']:8,.0f} sq ft")


Examples of area conversions:
10 Marla        ->    2,722 sq ft
4 Marla         ->    1,089 sq ft
5 Marla         ->    1,361 sq ft
1 Kanal         ->    5,445 sq ft
2 Kanal         ->   10,890 sq ft
17 Marla        ->    4,628 sq ft
8 Marla         ->    2,178 sq ft
3.1 Kanal       ->   16,880 sq ft
14.5 Marla      ->    3,948 sq ft
3 Marla         ->      817 sq ft
1.1 Kanal       ->    5,990 sq ft
3.5 Marla       ->      953 sq ft
5.2 Marla       ->    1,416 sq ft
5.8 Marla       ->    1,579 sq ft
11 Marla        ->    2,995 sq ft


In [23]:
# Optional: If you want to replace the original Area column with the numeric version
# Uncomment the following lines:

# df['Area'] = df['Area_SqFt']
# df = df.drop('Area_SqFt', axis=1)

# Or if you want to keep both columns, you can rename them for clarity:
# df = df.rename(columns={'Area': 'Area_Original', 'Area_SqFt': 'Area'})

print("DataFrame info after area conversion:")
print(df.info())


DataFrame info after area conversion:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7500 entries, 0 to 7499
Data columns (total 19 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Location                7500 non-null   object 
 1   Area                    7500 non-null   object 
 2   Baths                   7500 non-null   object 
 3   Beds                    7500 non-null   object 
 4   Dining Room             7500 non-null   int64  
 5   Laundry Room            7500 non-null   int64  
 6   Store Rooms             7500 non-null   int64  
 7   Kitchens                7500 non-null   int64  
 8   Drawing Room            7500 non-null   int64  
 9   Gym                     7500 non-null   int64  
 10  Powder Room             7500 non-null   int64  
 11  Steam Room              7500 non-null   int64  
 12  No additional rooms     7500 non-null   int64  
 13  Prayer Rooms            7500 non-null   int64  
 14  Lo

In [24]:
df.sample(5)

,Location,Area,Baths,Beds,Dining Room,Laundry Room,Store Rooms,Kitchens,Drawing Room,Gym,Powder Room,Steam Room,No additional rooms,Prayer Rooms,Lounge or Sitting Room,Price,Price_Numeric,Location_Encoded,Area_SqFt
56,"Canal Garden, Lahore, Punjab",5 Marla,6,5,1,0,0,2,1,0,1,0,0,0,1,PKR2.1 Crore,21000000.0,2.002273e+07,1361.25
7064,"Soan Garden, Islamabad, Islamabad Capital",12 Marla,6,7,1,0,2,2,1,0,0,1,0,0,1,PKR4.5 Crore,45000000.0,3.532771e+07,3267.00
5025,"DHA Defence, Islamabad, Islamabad Capital",1 Kanal,6,6,1,1,2,2,1,1,1,1,0,1,1,PKR11.5 Crore,115000000.0,9.447548e+07,5445.00
4750,"Bahria Town Karachi, Karachi, Sindh",200 Sq. Yd.,3,3,1,0,1,1,1,0,0,0,0,0,0,PKR1.77 Crore,17700000.0,1.959425e+07,1800.00
4871,"Bahria Town Karachi, Karachi, Sindh",79 Sq. Yd.,3,3,0,0,0,0,0,0,0,0,1,0,0,PKR71.23 Lakh,7123000.0,1.959425e+07,711.00


In [25]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

In [26]:
df = df.drop(["Price","Location","Area"], axis=1)

In [27]:
df = df.drop(["Beds","Baths"], axis=1)

In [28]:
df.sample(5)

,Dining Room,Laundry Room,Store Rooms,Kitchens,Drawing Room,Gym,Powder Room,Steam Room,No additional rooms,Prayer Rooms,Lounge or Sitting Room,Price_Numeric,Location_Encoded,Area_SqFt
6347,1,1,0,1,1,0,0,0,0,0,1,12500000.0,3.188200e+07,1361.25
3142,0,0,0,0,0,0,0,0,0,0,0,85000000.0,7.206316e+07,3150.00
6404,1,1,1,1,1,1,1,1,0,1,1,350000000.0,3.901180e+08,9801.00
2963,0,0,0,0,0,0,0,0,0,0,0,70000000.0,1.498673e+08,4500.00
7134,1,1,1,1,1,1,1,1,0,1,1,250000000.0,2.186163e+08,10890.00


In [29]:
model = LinearRegression()
scaler = StandardScaler()

In [30]:
x = df.drop("Price_Numeric", axis=1)

In [31]:
x.sample(5)

,Dining Room,Laundry Room,Store Rooms,Kitchens,Drawing Room,Gym,Powder Room,Steam Room,No additional rooms,Prayer Rooms,Lounge or Sitting Room,Location_Encoded,Area_SqFt
4932,1,0,1,1,1,0,0,0,0,0,1,1.498673e+08,2745.0
4819,1,1,1,2,1,1,1,1,0,1,1,1.959425e+07,1125.0
2552,1,1,2,2,1,0,1,0,0,1,1,7.789042e+07,1080.0
2194,0,0,0,2,0,0,0,0,0,0,0,8.656734e+07,5445.0
4759,1,1,2,2,1,1,1,1,0,1,1,1.498673e+08,13500.0


In [32]:
y = df["Price_Numeric"]

In [33]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [34]:
x.dtypes

Dining Room                 int64
Laundry Room                int64
Store Rooms                 int64
Kitchens                    int64
Drawing Room                int64
Gym                         int64
Powder Room                 int64
Steam Room                  int64
No additional rooms         int64
Prayer Rooms                int64
Lounge or Sitting Room      int64
Location_Encoded          float64
Area_SqFt                 float64
dtype: object

In [35]:
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

In [36]:
x_train_scaled[0:5]

array([[-1.28459203, -0.93760896, -0.64237755,  0.80996418,  0.73999212,
        -0.59043854, -1.00870455, -0.73326188, -0.4613635 , -0.83937206,
         0.91248049, -0.68393858, -0.74676861],
       [ 0.77845727, -0.93760896,  0.34110428,  1.77998115,  0.73999212,
        -0.59043854, -1.00870455, -0.73326188, -0.4613635 , -0.83937206,
         0.91248049, -0.68393858, -0.74676861],
       [ 0.77845727,  1.06654271, -0.64237755, -1.13006978,  0.73999212,
         1.69365639,  0.99137057,  1.36376924, -0.4613635 ,  1.19136679,
         0.91248049, -0.4171078 ,  0.47506514],
       [-1.28459203,  1.06654271,  0.34110428, -0.1600528 ,  0.73999212,
        -0.59043854,  0.99137057, -0.73326188, -0.4613635 , -0.83937206,
         0.91248049,  0.06043973, -0.66804221],
       [ 0.77845727, -0.93760896,  0.34110428, -0.1600528 ,  0.73999212,
        -0.59043854,  0.99137057, -0.73326188, -0.4613635 ,  1.19136679,
        -1.09591384,  0.06043973,  0.47506514]])

In [37]:
model.fit(x_train_scaled, y_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [38]:
y_pred = model.predict(x_test_scaled)

In [39]:
score = model.score(x_test_scaled, y_test)

In [40]:
score

0.6606978736579556

In [41]:
model_2 = LinearRegression()
model_2.fit(x_train, y_train)
y_pred_2 = model_2.predict(x_test)
score_2 = model_2.score(x_test, y_test)
score_2


0.6606978736334128

In [42]:
from sklearn.tree import DecisionTreeRegressor
model_3 = DecisionTreeRegressor()
model_3.fit(x_train, y_train)
y_pred_3 = model_3.predict(x_test)
score_3 = model_3.score(x_test, y_test)
score_3


0.7574852802575063

In [43]:
from sklearn.ensemble import RandomForestRegressor
model_4 = RandomForestRegressor()
model_4.fit(x_train, y_train)
y_pred_4 = model_4.predict(x_test)
score_4 = model_4.score(x_test, y_test)
score_4


0.8136935082981163

In [44]:
model_5 = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, max_samples=0.8, oob_score=True)
model_5.fit(x_train, y_train)
y_pred_5 = model_5.predict(x_test)
score_5 = model_5.score(x_test, y_test)
score_5

0.8260318747438228

In [47]:
from xgboost import XGBRegressor
model_5 = XGBRegressor()
model_5.fit(x_train, y_train)
y_pred_5 = model_5.predict(x_test)
score_5 = model_5.score(x_test, y_test)
score_5


0.7911620455431057